In [6]:
%pip install scikit-learn

  Obtaining dependency information for scikit-learn from https://files.pythonhosted.org/packages/9f/c4/0ab22726a04ede56f689476b760f98f8f46607caecff993017ac1b64aa5d/scikit_learn-1.8.0-cp312-cp312-win_amd64.whl.metadata
  Using cached scikit_learn-1.8.0-cp312-cp312-win_amd64.whl.metadata (11 kB)
  Obtaining dependency information for scipy>=1.10.0 from https://files.pythonhosted.org/packages/a2/84/dc08d77fbf3d87d3ee27f6a0c6dcce1de5829a64f2eae85a0ecc1f0daa73/scipy-1.17.1-cp312-cp312-win_amd64.whl.metadata
  Using cached scipy-1.17.1-cp312-cp312-win_amd64.whl.metadata (60 kB)
  Obtaining dependency information for joblib>=1.3.0 from https://files.pythonhosted.org/packages/7b/91/984aca2ec129e2757d1e4e3c81c3fcda9d0f85b74670a094cc443d9ee949/joblib-1.5.3-py3-none-any.whl.metadata
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Obtaining dependency information for threadpoolctl>=3.2.0 from https://files.pythonhosted.org/packages/32/d5/f9a850d79b0851d1d4ef6456097579a9005b31fea68


[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Define Features (X) and Target (y)
X = df[['LastPurchaseDaysAgo', 'TotalPurchases', 'TotalSpend', 'R', 'F', 'M']]
y = df['Churned']

# Split Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train the Model
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Predict and Evaluate
predictions = model.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, predictions):.2f}")
print(classification_report(y_test, predictions))

Accuracy: 0.78
              precision    recall  f1-score   support

           0       0.82      0.89      0.86      1036
           1       0.61      0.46      0.52       373

    accuracy                           0.78      1409
   macro avg       0.71      0.68      0.69      1409
weighted avg       0.76      0.78      0.77      1409



In [11]:
# 2. Calculate RFM Scores using Quantiles
quantiles = df[['LastPurchaseDaysAgo', 'TotalPurchases', 'TotalSpend']].quantile(q=[0.25, 0.5, 0.75]).to_dict()

def r_score(x, p, d):
    if x <= d[p][0.25]: return 4 # Bought recently = Good (4)
    elif x <= d[p][0.50]: return 3
    elif x <= d[p][0.75]: return 2
    else: return 1

def fm_score(x, p, d):
    if x <= d[p][0.25]: return 1 # Spent little/Bought rarely = Bad (1)
    elif x <= d[p][0.50]: return 2
    elif x <= d[p][0.75]: return 3
    else: return 4

df['R'] = df['LastPurchaseDaysAgo'].apply(r_score, args=('LastPurchaseDaysAgo', quantiles,))
df['F'] = df['TotalPurchases'].apply(fm_score, args=('TotalPurchases', quantiles,))
df['M'] = df['TotalSpend'].apply(fm_score, args=('TotalSpend', quantiles,))

# Combine into one RFM Score string
df['RFM_Segment'] = df['R'].map(str) + df['F'].map(str) + df['M'].map(str)
df.head()

,LastPurchaseDaysAgo,TotalPurchases,TotalSpend,Churned,R,F,M,RFM_Segment
0,39,1,29.85,0,2,1,1,211
1,52,34,56.95,0,1,3,2,132
2,29,2,53.85,1,3,1,2,312
3,15,45,42.30,0,4,3,2,432
4,43,2,70.70,1,2,1,3,213


In [8]:
%pip install joblib

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
import pandas as pd
import numpy as np

# 1. Load the real Kaggle dataset
df = pd.read_csv('telecom_data.csv')

# 2. Let's make the data match our RFM structure!
# We will rename the real columns to match what our model expects.
df = df.rename(columns={
    'tenure': 'TotalPurchases', # How many months they stayed
    'MonthlyCharges': 'TotalSpend' # How much they pay
})

# 3. Create a fake 'LastPurchaseDaysAgo' since telecom doesn't usually track this
np.random.seed(42)
df['LastPurchaseDaysAgo'] = np.random.randint(1, 60, size=len(df))

# 4. Convert the text target ('Yes'/'No') into computer numbers (1 and 0)
df['Churned'] = df['Churn'].apply(lambda x: 1 if x == 'Yes' else 0)

# Keep only the columns we need
df = df[['LastPurchaseDaysAgo', 'TotalPurchases', 'TotalSpend', 'Churned']]
df.head()

,LastPurchaseDaysAgo,TotalPurchases,TotalSpend,Churned
0,39,1,29.85,0
1,52,34,56.95,0
2,29,2,53.85,1
3,15,45,42.30,0
4,43,2,70.70,1


In [13]:
import joblib
joblib.dump(model, 'churn_model.pkl')
print("Real model saved successfully!")

Real model saved successfully!
